In [1]:
# Parameters (papermill overrides these via -p DATA_DIR ... -p SEED ... -p OUT_DIR ...)
DATA_DIR = "./data"
SEED = 42
OUT_DIR = "reports"

In [2]:
# Parameters
DATA_DIR = "./data"
SEED = 42
OUT_DIR = "reports"


# 04 — Comparación entre corpus
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Vocabulario distintivo por fuente, índice de Jaccard. Validez externa del merge.

**Nota:** Los análisis multivariable (Jaccard entre fuentes, log-odds) requieren ≥2 fuentes. Con una sola fuente, este notebook se reduce a top-N vocabulario + estadísticas descriptivas.


## Parámetros (papermill)
- `DATA_DIR`: ruta a `data/` (default `./data`).
- `SEED`: semilla (default 42).
- `OUT_DIR`: dónde guardar figuras y tablas (default `reports`).


In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

DATA_DIR = Path(os.environ.get("DATA_DIR", DATA_DIR))
SEED = int(os.environ.get("SEED", SEED))
OUT_DIR = Path(os.environ.get("OUT_DIR", OUT_DIR))
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

# Carga con fallback.
processed = DATA_DIR / "processed" / "corpus_v1.parquet"
if processed.exists():
    df = pd.read_parquet(processed)
else:
    frames = [pd.read_parquet(p) for p in (DATA_DIR / "interim").glob("*/data.parquet")]
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"corpus: {len(df):,} filas, {df['source'].nunique()} fuentes ({sorted(df['source'].unique())})")

N_SOURCES = df["source"].nunique()
sources = sorted(df["source"].unique())

corpus: 1,047,194 filas, 1 fuentes (['coello_guilarte'])


In [4]:
# Top-20 vocabulario por fuente.
def vocab_top(texts, top=20):
    toks = Counter()
    for t in texts:
        toks.update((t or "").lower().split())
    return toks.most_common(top)

tables = {}
for src in sources:
    sub = df[df["source"] == src]
    top = vocab_top(sub["text_clean"].fillna(""), top=20)
    tables[src] = pd.DataFrame(top, columns=["term", "count"])
    print(f"--- {src} (n={len(sub):,}) ---")
    for w, c in top:
        print(f"  {w:20s} {c:,}")
    print()

# Persistir top-20 por fuente.
for src, t in tables.items():
    t.to_csv(OUT_DIR / "tables" / f"eda_04_top20_vocab_{src}.csv", index=False)

--- coello_guilarte (n=1,047,194) ---
  de                   444,198
  :                    381,837
  que                  374,321
  a                    271,335
  la                   268,896
  y                    232,082
  en                   219,949
  el                   207,818
  no                   193,028
  me                   164,152
  es                   117,859
  un                   104,550
  se                   99,788
  con                  97,813
  por                  95,744
  los                  95,078
  te                   92,702
  lo                   91,904
  mi                   84,184
  para                 80,253



In [5]:
# Jaccard entre vocabularios (requiere ≥2 fuentes).
def v_set(texts, min_freq=2):
    toks = Counter()
    for t in texts:
        toks.update((t or "").lower().split())
    return {w for w, c in toks.items() if c >= min_freq}

mat = None
if N_SOURCES >= 2:
    vocabs = {s: v_set(df[df["source"] == s]["text_clean"].fillna("")) for s in sources}
    mat = pd.DataFrame(index=sources, columns=sources, dtype=float)
    for a in sources:
        for b in sources:
            u = len(vocabs[a] | vocabs[b])
            mat.loc[a, b] = len(vocabs[a] & vocabs[b]) / u if u else 0

    print(mat)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(mat.astype(float), annot=True, fmt=".2f", cmap="Greens", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "Jaccard"})
    ax.set_title("Jaccard entre vocabularios de fuentes (min_freq=2)")
    plt.tight_layout()
    out = OUT_DIR / "figures" / "eda_04_jaccard_fuentes.png"
    plt.savefig(out, dpi=120)
    plt.show()
    print(f"figura -> {out}")
    mat.to_csv(OUT_DIR / "tables" / "eda_04_jaccard_matrix.csv")
else:
    print(f"AVISO: solo hay {N_SOURCES} fuente ({sources}); Jaccard entre fuentes requiere >=2.")
    print("Saltando heatmap Jaccard. Análisis multivariable se omite para este corpus.")

AVISO: solo hay 1 fuente (['coello_guilarte']); Jaccard entre fuentes requiere >=2.
Saltando heatmap Jaccard. Análisis multivariable se omite para este corpus.


In [6]:
# Log-odds por fuente (requiere ≥2 fuentes).
def log_odds(src_texts, other_texts, top=15):
    a, b = Counter(), Counter()
    for t in src_texts:
        a.update((t or "").lower().split())
    for t in other_texts:
        b.update((t or "").lower().split())
    all_words = set(a) | set(b)
    N_a, N_b = sum(a.values()), sum(b.values())
    out = []
    for w in all_words:
        if a[w] + b[w] < 5:
            continue
        p_a = (a[w] + 1) / (N_a + len(all_words))
        p_b = (b[w] + 1) / (N_b + len(all_words))
        out.append((w, np.log(p_a / p_b)))
    out.sort(key=lambda x: -x[1])
    return out[:top], out[-top:]

if N_SOURCES >= 2:
    logodds_tables = {}
    for src in sources:
        sub = df[df["source"] == src]["text_clean"].fillna("").tolist()
        other = df[df["source"] != src]["text_clean"].fillna("").tolist()
        up, down = log_odds(sub, other, top=15)
        logodds_tables[src] = pd.DataFrame({
            "mas_en_fuente": [w for w, _ in up],
            "logodds_up": [round(v, 3) for _, v in up],
            "menos_en_fuente": [w for w, _ in down],
            "logodds_down": [round(v, 3) for _, v in down],
        })
        print(f"--- {src} ---")
        print(f"  mas en esta fuente: {[w for w, _ in up]}")
        print(f"  menos en esta fuente: {[w for w, _ in down]}")
        print()
    for src, t in logodds_tables.items():
        t.to_csv(OUT_DIR / "tables" / f"eda_04_logodds_{src}.csv", index=False)
else:
    print(f"AVISO: con {N_SOURCES} fuente no se puede computar log-odds 'una vs resto'.")

AVISO: con 1 fuente no se puede computar log-odds 'una vs resto'.


In [7]:
# Conclusiones dinámicas.
from IPython.display import Markdown, display

if N_SOURCES >= 2:
    _offdiag = []
    for i, a in enumerate(sources):
        for b in sources[i+1:]:
            _offdiag.append(float(mat.loc[a, b]))
    _mean_j = np.mean(_offdiag) if _offdiag else 0
    _md = f"""
## Conclusiones

- **{N_SOURCES} fuentes activas**: {sources}.
- **Jaccard off-diagonal medio**: {_mean_j:.3f}.
  - < 0.10 → fuentes disjuntas (poco merge útil).
  - 0.20–0.40 → rango saludable.
  - > 0.50 → fuentes redundantes.
- **Log-odds por fuente**: tablas exportadas a CSV en `reports/tables/eda_04_logodds_*.csv`.
"""
else:
    _md = f"""
## Conclusiones

- **Solo {N_SOURCES} fuente activa**: {sources}. Los análisis multivariable (Jaccard entre fuentes, log-odds) se omiten automáticamente porque requieren ≥2 corpus.
- **Top-20 vocabulario ({sources[0]})**: predominan stopwords (`de`, `que`, `a`, `la`) y pronombres (`me`, `te`, `mi`).
- **Limitación**: no se puede medir la validez externa del merge con un solo corpus. Para análisis multivariable: descargar al menos una de las fuentes funcionales (`redsm5_sample`, `emoevales`, `swmh_es` desde HuggingFace).
"""
display(Markdown(_md))



## Conclusiones

- **Solo 1 fuente activa**: ['coello_guilarte']. Los análisis multivariable (Jaccard entre fuentes, log-odds) se omiten automáticamente porque requieren ≥2 corpus.
- **Top-20 vocabulario (coello_guilarte)**: predominan stopwords (`de`, `que`, `a`, `la`) y pronombres (`me`, `te`, `mi`).
- **Limitación**: no se puede medir la validez externa del merge con un solo corpus. Para análisis multivariable: descargar al menos una de las fuentes funcionales (`redsm5_sample`, `emoevales`, `swmh_es` desde HuggingFace).
